# LPSE-X Offline Inference Readiness Notebook

Notebook ini adalah jalur inferensi final-round: memakai artefak submission `model_risk.ubj` / `model_risk.onnx`, membaca full held-out split `test_data/features.parquet`, dan tidak menjalankan training, HPO, scraping, cloud call, atau export model baru.

Output harus dibaca sebagai **triase risiko** dan **prioritas review**, **bukan tuduhan pelanggaran**.


In [ ]:
from pathlib import Path
import pandas as pd

from src.product_demo import SAFE_GUARDRAIL_ID, build_inference_run

FEATURE_SOURCE = Path('test_data/features.parquet')
RAW_SOURCE = Path('test_data/raw.parquet')
UBJ_ARTIFACT = Path('model_risk.ubj')
ONNX_ARTIFACT = Path('model_risk.onnx')

assert FEATURE_SOURCE.exists(), 'test_data/features.parquet wajib tersedia untuk inferensi lokal'
assert RAW_SOURCE.exists(), 'test_data/raw.parquet wajib tersedia untuk metadata paket'
assert UBJ_ARTIFACT.exists(), 'model_risk.ubj wajib tersedia; notebook ini tidak retraining model'
assert ONNX_ARTIFACT.exists(), 'model_risk.onnx wajib tersedia untuk smoke check artefak submission'

SAFE_GUARDRAIL_ID


In [ ]:
# Full held-out inference: max_rows=None berarti seluruh test_data/features.parquet discore.
dataset, backend, predictions, queue, metadata = build_inference_run(max_rows=None, top_n=50)

status = {
    'model_artifact': metadata.model_artifact,
    'backend': metadata.model_backend,
    'mode': metadata.inference_mode,
    'feature_source': metadata.feature_source,
    'raw_source': metadata.raw_source,
    'rows_scored': metadata.rows_scored,
    'rows_ranked': metadata.rows_ranked,
    'rows_displayed': metadata.rows_displayed,
    'latency_ms': metadata.total_latency_ms,
    'no_cloud_call': metadata.no_cloud_call,
    'no_retraining': metadata.no_retraining,
    'no_live_scraping': metadata.no_live_scraping,
}

assert metadata.model_artifact == 'model_risk.ubj'
assert metadata.feature_source == 'test_data/features.parquet'
assert metadata.raw_source == 'test_data/raw.parquet'
assert metadata.rows_scored == len(pd.read_parquet(FEATURE_SOURCE))
assert metadata.rows_ranked == len(predictions) == len(dataset.features)
assert metadata.rows_displayed == len(queue) == 50
assert metadata.no_cloud_call and metadata.no_retraining and metadata.no_live_scraping

status


## Ranked review queue

Dashboard sengaja menampilkan Top 50 agar reviewer fokus. Ini bukan berarti hanya 50 tender yang discore; `rows_scored` di atas menunjukkan seluruh held-out split sudah melewati model lokal.


In [ ]:
queue.loc[:, [
    'risk_rank',
    'case_id',
    'package_title',
    'buyer',
    'supplier',
    'predicted_label',
    'risk_priority_score',
]].head(10)


In [ ]:
from src.casebook import build_casebook

top_case_id = str(queue.iloc[0]['case_id'])
casebook = build_casebook(top_case_id, dataset, predictions, backend)

casebook_summary = {
    'case_id': casebook['case_id'],
    'predicted_label': casebook['model_output']['predicted_label'],
    'probability': casebook['model_output']['probability'],
    'risk_rank': casebook['model_output']['risk_rank'],
    'top_factors': [factor['feature_label'] for factor in casebook['factors'][:5]],
    'guardrail': casebook['guardrail'],
}

assert casebook_summary['risk_rank'] == 1
assert 'bukan tuduhan pelanggaran' in casebook_summary['guardrail']
casebook_summary


In [ ]:
# Optional ONNX smoke check over the same aligned feature order.
import numpy as np
import onnxruntime as rt

session = rt.InferenceSession(str(ONNX_ARTIFACT), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
sample = dataset.features.loc[:, backend.feature_names].head(5).to_numpy(dtype=np.float32)
onnx_outputs = session.run(None, {input_name: sample})

assert len(onnx_outputs) > 0
[getattr(output, 'shape', type(output).__name__) for output in onnx_outputs]


## Operational notes

- Jalur ini hanya inferensi lokal atas split `test_data`.
- Tidak ada panggilan training/fitting model, tuning, scraping, cloud call, atau penulisan ulang artefak model.
- Untuk produk demo, metadata yang sama diekspos melalui FastAPI: `/api/demo-state`, `/api/queue`, dan `/api/inference-status`.
